# Real Estate Investment Advisor
## Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/india_housing_prices.csv")
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicates ignoring ID:", df.drop(columns="ID").duplicated().sum())
print("Floor_No > Total_Floors:", (df["Floor_No"] > df["Total_Floors"]).sum())
print("Age matches 2025 - Year_Built:",
      ((2025 - df["Year_Built"]) == df["Age_of_Property"]).all())

Shape: (250000, 23)
Missing values: 0
Duplicate rows: 0
Duplicates ignoring ID: 0
Floor_No > Total_Floors: 116304
Age matches 2025 - Year_Built: True


### Data quality findings
- **Missing values:** none. **Duplicates:** none, even when ID is ignored.
- **Price_per_SqFt** is stored in lakhs and rounded to 2 decimals, so it loses
  precision (some values show 0.0). It is recalculated in rupees.
- **Floor_No exceeds Total_Floors** in 116,304 rows (47%). Deleting these would
  remove almost half the data, so Floor_No is capped at Total_Floors, and a
  flag column records which rows were corrected.

In [2]:
# Recalculating price per sq ft in rupees (1 lakh = 100,000)
df["Price_per_SqFt"] = (df["Price_in_Lakhs"] * 100000 / df["Size_in_SqFt"]).round(2)

# Flagging and fixing impossible floor numbers
df["Floor_Inconsistent"] = (df["Floor_No"] > df["Total_Floors"]).astype(int)
df["Floor_No"] = np.minimum(df["Floor_No"], df["Total_Floors"])

display(df[["Price_in_Lakhs", "Size_in_SqFt", "Price_per_SqFt",
    "Floor_No", "Total_Floors", "Floor_Inconsistent"]].head(3))

print("Rows flagged:", df["Floor_Inconsistent"].sum())

,Price_in_Lakhs,Size_in_SqFt,Price_per_SqFt,Floor_No,Total_Floors,Floor_Inconsistent
0,489.76,4740,10332.49,1,1,1
1,195.52,2364,8270.73,20,20,1
2,183.79,3642,5046.40,19,27,0


Rows flagged: 116304


### Feature engineering
- **Amenity_Count:** number of amenities listed, plus one 0/1 column per amenity.
- **School_Density_Score:** nearby schools scaled to 0–1.
- **Infrastructure_Score:** average of schools, hospitals and transport access
  (Low = 2, Medium = 6, High = 10), on a 1–10 scale.

In [3]:
df["Amenity_Count"] = df["Amenities"].str.split(",").str.len()
for a in ["Gym", "Pool", "Clubhouse", "Garden", "Playground"]:
    df[f"Has_{a}"] = df["Amenities"].str.contains(a).astype(int)

transport_score = df["Public_Transport_Accessibility"].map(
    {"Low": 2, "Medium": 6, "High": 10})
df["School_Density_Score"] = df["Nearby_Schools"] / 10
df["Infrastructure_Score"] = ((df["Nearby_Schools"] + df["Nearby_Hospitals"]
                               + transport_score) / 3).round(2)

df[["Amenity_Count", "Has_Gym", "Has_Pool",
    "School_Density_Score", "Infrastructure_Score"]].head(3)

,Amenity_Count,Has_Gym,Has_Pool,School_Density_Score,Infrastructure_Score
0,5,1,1,1.0,7.67
1,5,1,1,0.8,3.67
2,4,1,1,0.9,6.33


### Target 1 (Regression): Future Price after 5 years
Future Price = Current Price × (1 + r)^5, where the yearly growth rate r starts
at 8% and is adjusted by property features:
- Public transport: High +1%, Low −1%
- Property age: 10 years or less +1%, over 25 years −1%
- 4 or more amenities: +0.5%
- Security available: +0.5%

This gives growth rates between 6% and 11% per year.

In [4]:
rate = 0.08
rate = rate + df["Public_Transport_Accessibility"].map(
    {"High": 0.01, "Medium": 0.0, "Low": -0.01})
rate = rate + np.select([df["Age_of_Property"] <= 10, df["Age_of_Property"] > 25],
                        [0.01, -0.01], 0)
rate = rate + np.where(df["Amenity_Count"] >= 4, 0.005, 0)
rate = rate + np.where(df["Security"] == "Yes", 0.005, 0)

df["Growth_Rate"] = rate
df["Future_Price_5Y"] = (df["Price_in_Lakhs"] * (1 + df["Growth_Rate"]) ** 5).round(2)
df["Growth_Rate"].describe()

count    250000.000000
mean          0.084248
std           0.011636
min           0.060000
25%           0.075000
50%           0.085000
75%           0.095000
max           0.110000
Name: Growth_Rate, dtype: float64

### Target 2 (Classification): Good Investment
A multi-factor score gives 1 point for each condition met:
1. Price per sq ft at or below the city median (good value)
2. BHK of 3 or more
3. Ready to move (no construction risk)
4. Public transport access High or Medium
5. 5 or more nearby schools

A property scoring **3 or more out of 5** is labelled a Good Investment (1).

In [5]:
city_median_pps = df.groupby("City")["Price_per_SqFt"].transform("median")

df["Investment_Score"] = (
    (df["Price_per_SqFt"] <= city_median_pps).astype(int)
    + (df["BHK"] >= 3).astype(int)
    + (df["Availability_Status"] == "Ready_to_Move").astype(int)
    + df["Public_Transport_Accessibility"].isin(["High", "Medium"]).astype(int)
    + (df["Nearby_Schools"] >= 5).astype(int)
)
df["Good_Investment"] = (df["Investment_Score"] >= 3).astype(int)
df["Good_Investment"].value_counts(normalize=True)

Good_Investment
1    0.636392
0    0.363608
Name: proportion, dtype: float64

In [6]:
df.to_csv("../data/cleaned_housing_data.csv", index=False)
print("Saved:", df.shape)

Saved: (250000, 36)
